In [5]:
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever

raw_documents = [
    Document(page_content="보안 규정: 업무용 PC는 3개월마다 비밀번호를 변경해야 하며 특수문자를 포함해야 합니다."),
    Document(page_content="원격 근무 가이드: 재택 근무 시 반드시 사내 VPN을 연결하여 사내 인트라넷에 접속해야 합니다."),
    Document(page_content="장비 지급 규정: 신규 입사자에게는 노트북 1대와 외장 모니터 1대가 기본 지급됩니다."),
    Document(page_content="계정 관리: 사내 계정 5회 연속 로그인 실패 시 계정이 잠기며 IT 지원팀에 잠금 해제를 요청해야 합니다.")
]

def custom_tokenizer(text):
    return text.split()

bm25_retriever = BM25Retriever.from_documents(
    documents=raw_documents,
    preprocess_func=custom_tokenizer
)

bm25_retriever.k = 3



In [6]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_classic.retrievers import ContextualCompressionRetriever

cross_encoder_model = HuggingFaceCrossEncoder(model_name="Dongjin-kr/ko-reranker")

reranker = CrossEncoderReranker(
    model=cross_encoder_model,
    top_n=2
)

rerank_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=bm25_retriever
)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 540.08it/s]


In [7]:
query = "사내 망 들어가려다 비밀번호를 여러 번 틀렸을 때 처리 방법"

print("=== BM25 ===")
bm25_docs = bm25_retriever.invoke(query)
for i, doc in enumerate(bm25_docs, 1):
    print(f"Rank:{i}, Doc: {doc.page_content}")

print("=== ReRanker ===")
rerank_docs = rerank_retriever.invoke(query)
for i, doc in enumerate(rerank_docs, 1):
    print(f"Rank:{i}, Doc: {doc.page_content}")

=== BM25 ===
Rank:1, Doc: 보안 규정: 업무용 PC는 3개월마다 비밀번호를 변경해야 하며 특수문자를 포함해야 합니다.
Rank:2, Doc: 계정 관리: 사내 계정 5회 연속 로그인 실패 시 계정이 잠기며 IT 지원팀에 잠금 해제를 요청해야 합니다.
Rank:3, Doc: 장비 지급 규정: 신규 입사자에게는 노트북 1대와 외장 모니터 1대가 기본 지급됩니다.
=== ReRanker ===
Rank:1, Doc: 계정 관리: 사내 계정 5회 연속 로그인 실패 시 계정이 잠기며 IT 지원팀에 잠금 해제를 요청해야 합니다.
Rank:2, Doc: 보안 규정: 업무용 PC는 3개월마다 비밀번호를 변경해야 하며 특수문자를 포함해야 합니다.
